# ⚡ Piecewise Linear Attention - Performance Benchmark

Comprehensive benchmark comparing **Standard**, **Linear**, and **Piecewise** attention mechanisms.

## What This Does

- Benchmarks pure attention performance (without transformer overhead)
- Measures speed, accuracy, and speedup across different scales
- Runs scaling analysis to show how speedup increases with sequence length
- Visualizes results with plots

## Expected Runtime

- **Quick benchmark**: ~5-10 minutes
- **Full benchmark with scaling**: ~10-15 minutes

---

## 📦 Setup

In [ ]:
# Configuration
REPO_URL = "https://github.com/grapentt/piecewise-linear-attention.git"
BRANCH = "main"  # Change to your branch if needed

# Clone repository
import os
if not os.path.exists("piecewise-linear-attention"):
    print(f"Cloning repository (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already exists")

%cd piecewise-linear-attention
!git pull origin {BRANCH}
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
!pip install -e ".[benchmark]" -q
print("✓ Installation complete!")

In [ ]:
# Check device
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("⚠ No GPU - using CPU (slower but still works)")

print(f"\nDevice: {device}")

---

## 🎯 Benchmark Configuration

In [ ]:
# Benchmark settings
BENCHMARK_TYPE = "basic"  # Options: "basic", "scaling"
CAUSAL = False  # Set to True to benchmark causal masking
NUM_RUNS = 20  # Number of runs per configuration (more = more reliable)

print("Benchmark Configuration:")
print(f"  Type: {BENCHMARK_TYPE}")
print(f"  Causal masking: {CAUSAL}")
print(f"  Runs per config: {NUM_RUNS}")
print(f"  Device: {device}")

---

## 🚀 Run Benchmark

In [ ]:
# Run benchmark
!python benchmarks/benchmark_attention.py \
  {"--causal" if CAUSAL else ""} \
  {"--scale" if BENCHMARK_TYPE == "scaling" else ""} \
  --num-runs {NUM_RUNS} \
  --save results/benchmark_results.json

print("\n" + "="*80)
print("✓ Benchmark complete!")
print("="*80)

---

## 📊 Visualize Results

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load results
with open("results/benchmark_results.json") as f:
    results = json.load(f)

print(f"Loaded {len(results)} benchmark results")

In [ ]:
if BENCHMARK_TYPE == "scaling":
    # Scaling analysis plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    seq_lens = [r["seq_len"] for r in results]
    std_times = [r["standard_time_ms"] for r in results]
    lin_times = [r["linear_time_ms"] for r in results]
    pw_times = [r["piecewise_time_ms"] for r in results]
    
    # Plot 1: Absolute times
    ax = axes[0]
    ax.plot(seq_lens, std_times, 'o-', linewidth=2, markersize=8, label='Standard', color='#e74c3c')
    ax.plot(seq_lens, lin_times, 's-', linewidth=2, markersize=8, label='Linear', color='#3498db')
    ax.plot(seq_lens, pw_times, '^-', linewidth=2, markersize=8, label='Piecewise', color='#2ecc71')
    ax.set_xlabel('Sequence Length', fontsize=12, fontweight='bold')
    ax.set_ylabel('Time (ms)', fontsize=12, fontweight='bold')
    ax.set_title('Execution Time vs Sequence Length', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    
    # Plot 2: Speedup over standard
    ax = axes[1]
    lin_speedups = [r["linear_speedup"] for r in results]
    pw_speedups = [r["piecewise_speedup"] for r in results]
    
    ax.plot(seq_lens, lin_speedups, 's-', linewidth=2, markersize=8, label='Linear', color='#3498db')
    ax.plot(seq_lens, pw_speedups, '^-', linewidth=2, markersize=8, label='Piecewise', color='#2ecc71')
    ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Baseline')
    ax.set_xlabel('Sequence Length', fontsize=12, fontweight='bold')
    ax.set_ylabel('Speedup vs Standard', fontsize=12, fontweight='bold')
    ax.set_title('Speedup Scaling Analysis', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log', base=2)
    
    fig.suptitle(f"Scaling Analysis - {'Causal' if CAUSAL else 'Non-Causal'} Attention", 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Print key insights
    print("\n" + "="*80)
    print("KEY INSIGHTS")
    print("="*80)
    print(f"At n={seq_lens[0]}: Piecewise is {pw_speedups[0]:.1f}× faster")
    print(f"At n={seq_lens[-1]}: Piecewise is {pw_speedups[-1]:.1f}× faster")
    print(f"\n✨ Speedup increases {pw_speedups[-1]/pw_speedups[0]:.1f}× as sequence length grows!")
    print("="*80)
    
else:
    # Basic benchmark plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Extract data
    configs = [f"b{r['batch']}_n{r['seq_len']}_d{r['dim']}" for r in results]
    std_times = [r["standard"]["mean_time_ms"] for r in results]
    lin_times = [r["linear"]["mean_time_ms"] for r in results]
    pw_times = [r["piecewise"]["mean_time_ms"] for r in results]
    lin_speedups = [r["linear"]["speedup"] for r in results]
    pw_speedups = [r["piecewise"]["speedup"] for r in results]
    lin_errors = [r["linear"]["relative_error"] * 100 for r in results]
    pw_errors = [r["piecewise"]["relative_error"] * 100 for r in results]
    
    x = np.arange(len(configs))
    width = 0.25
    
    # Plot 1: Absolute times
    ax = axes[0, 0]
    ax.bar(x - width, std_times, width, label='Standard', color='#e74c3c')
    ax.bar(x, lin_times, width, label='Linear', color='#3498db')
    ax.bar(x + width, pw_times, width, label='Piecewise', color='#2ecc71')
    ax.set_ylabel('Time (ms)', fontsize=11, fontweight='bold')
    ax.set_title('Execution Time Comparison', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(configs, rotation=45, ha='right', fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Speedup
    ax = axes[0, 1]
    ax.bar(x - width/2, lin_speedups, width, label='Linear', color='#3498db')
    ax.bar(x + width/2, pw_speedups, width, label='Piecewise', color='#2ecc71')
    ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, linewidth=2)
    ax.set_ylabel('Speedup vs Standard', fontsize=11, fontweight='bold')
    ax.set_title('Speedup Comparison', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(configs, rotation=45, ha='right', fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Relative error
    ax = axes[1, 0]
    ax.bar(x - width/2, lin_errors, width, label='Linear', color='#3498db')
    ax.bar(x + width/2, pw_errors, width, label='Piecewise', color='#2ecc71')
    ax.set_ylabel('Relative Error (%)', fontsize=11, fontweight='bold')
    ax.set_title('Approximation Error', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(configs, rotation=45, ha='right', fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Speedup vs Error tradeoff
    ax = axes[1, 1]
    ax.scatter(lin_errors, lin_speedups, s=100, alpha=0.7, label='Linear', color='#3498db')
    ax.scatter(pw_errors, pw_speedups, s=100, alpha=0.7, label='Piecewise', color='#2ecc71')
    ax.set_xlabel('Relative Error (%)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Speedup vs Standard', fontsize=11, fontweight='bold')
    ax.set_title('Speed-Accuracy Tradeoff', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    fig.suptitle(f"Attention Mechanism Benchmark - {'Causal' if CAUSAL else 'Non-Causal'}",
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    print(f"\nAverage speedup:")
    print(f"  Linear:    {np.mean(lin_speedups):.2f}×")
    print(f"  Piecewise: {np.mean(pw_speedups):.2f}×")
    print(f"\nAverage error:")
    print(f"  Linear:    {np.mean(lin_errors):.2f}%")
    print(f"  Piecewise: {np.mean(pw_errors):.2f}%")
    print(f"\n✨ Piecewise is {np.mean(lin_errors) - np.mean(pw_errors):.1f}% more accurate than Linear!")
    print("="*80)

---

## 📋 Detailed Results Table

In [ ]:
import pandas as pd

if BENCHMARK_TYPE == "scaling":
    # Scaling table
    df_data = []
    for r in results:
        df_data.append({
            "Seq Length": r["seq_len"],
            "Standard (ms)": f"{r['standard_time_ms']:.2f}",
            "Linear (ms)": f"{r['linear_time_ms']:.2f}",
            "Piecewise (ms)": f"{r['piecewise_time_ms']:.2f}",
            "Linear Speedup": f"{r['linear_speedup']:.2f}×",
            "Piecewise Speedup": f"{r['piecewise_speedup']:.2f}×",
        })
    df = pd.DataFrame(df_data)
    print("\nScaling Analysis Results:")
    print(df.to_string(index=False))
else:
    # Basic benchmark table
    df_data = []
    for r in results:
        config = f"b{r['batch']}, n{r['seq_len']}, d{r['dim']}"
        df_data.append({
            "Config": config,
            "Standard (ms)": f"{r['standard']['mean_time_ms']:.2f}",
            "Linear (ms)": f"{r['linear']['mean_time_ms']:.2f}",
            "Linear Speedup": f"{r['linear']['speedup']:.2f}×",
            "Linear Error": f"{r['linear']['relative_error']*100:.2f}%",
            "Piecewise (ms)": f"{r['piecewise']['mean_time_ms']:.2f}",
            "PW Speedup": f"{r['piecewise']['speedup']:.2f}×",
            "PW Error": f"{r['piecewise']['relative_error']*100:.2f}%",
        })
    df = pd.DataFrame(df_data)
    print("\nBenchmark Results:")
    print(df.to_string(index=False))

---

## 💾 Download Results

In [ ]:
from google.colab import files

print("Downloading results...")
files.download("results/benchmark_results.json")
print("✓ Download complete!")

---

## 🎛️ Advanced: Custom Benchmark

Run a custom benchmark with specific configurations.

In [ ]:
# Example: Benchmark causal attention with scaling analysis
# Uncomment and run:

# !python benchmarks/benchmark_attention.py \
#   --causal \
#   --scale \
#   --num-runs 10 \
#   --save results/causal_scaling.json

---

## 📚 Learn More

- [Main README](https://github.com/grapentt/piecewise-linear-attention/blob/main/README.md)
- [Theory](https://github.com/grapentt/piecewise-linear-attention/blob/main/THEORY.md)
- [Benchmark README](https://github.com/grapentt/piecewise-linear-attention/blob/main/benchmarks/README.md)

**Happy Benchmarking! 🚀**